<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-09-measure-and-improve-the-meridian-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9 (graded) — Measure and improve the Meridian assistant
**Course 2: Generative AI and LLMs with Python — Chapter 9: RAG II: evaluation & advanced retrieval**

**Two weeks later — Leo Farkas:** "The assistant sounds confident but compliance caught two
wrong answers. How do we measure whether it's trustworthy, and make retrieval better?"

**What you'll submit:** a RAGAS scorecard, ≥ 2 retrieval upgrades with a measured lift, and
an iterative-retrieval loop for multi-hop questions.

In [ ]:
!pip install -q sentence-transformers faiss-cpu ragas datasets

## 1. Rebuild the Chapter 8 corpus + baseline retriever (compact version)

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

documents = [
    ('kyc-001', 'Customer Identification Program', 'Meridian Bank requires a government-issued photo ID and proof of address for every new account. Accounts may not transact until identity verification is complete. Enhanced due diligence applies to higher-risk accounts, including politically exposed persons.'),
    ('aml-002', 'Suspicious Activity Reporting', 'A Suspicious Activity Report (SAR) is filed for transactions meeting regulatory thresholds, such as structuring deposits under $10,000 to avoid reporting, within 24 hours of detection by the AML team.'),
    ('ege-003', 'Reg E Error Resolution', 'Customers have 60 days from the statement date to report an unauthorized electronic transfer. The bank must resolve the claim within 10 business days, or provisionally credit the account for up to 45 days in complex cases. Liability is capped at $50 if reported within 2 business days.'),
    ('od-004', 'Overdraft Protection Policy', 'Customers must opt in to overdraft coverage for debit card transactions. Overdraft fees are capped at 3 per day. Accounts overdrawn more than 60 consecutive days are referred to collections.'),
    ('lend-005', 'Fair Lending & Adverse Action', 'A denied credit application requires an adverse action notice within 30 days stating the specific reasons. Race, religion, and marital status may never factor into a credit decision.'),
    ('card-007', 'Credit Card Dispute Process', 'Cardholders have 60 days to dispute a billing error. The bank must acknowledge within 30 days and resolve within two billing cycles, not exceeding 90 days. No late fee accrues on the disputed amount while under investigation.'),
]

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [d[2] for d in documents]
chunk_titles = [d[1] for d in documents]
chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)

def baseline_retrieve(query, top_k=2):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    sims = (chunk_embeddings @ q_emb.T).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return [{'title': chunk_titles[i], 'text': chunk_texts[i], 'score': float(sims[i])} for i in top_idx]

## 2. Upgrade 1: reranking with a cross-encoder

In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_with_rerank(query, wide_k=6, final_k=2):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    sims = (chunk_embeddings @ q_emb.T).ravel()
    wide_idx = np.argsort(-sims)[:wide_k]
    pairs = [(query, chunk_texts[i]) for i in wide_idx]
    rerank_scores = reranker.predict(pairs)
    order = np.argsort(-rerank_scores)[:final_k]
    chosen = [wide_idx[o] for o in order]
    return [{'title': chunk_titles[i], 'text': chunk_texts[i], 'score': float(rerank_scores[order[j]])}
            for j, i in enumerate(chosen)]

q = 'What happens if I report a fraudulent transfer right away?'
print('Baseline:', [r['title'] for r in baseline_retrieve(q)])
print('Reranked:', [r['title'] for r in retrieve_with_rerank(q)])

## 3. Upgrade 2: HyDE (hypothetical document embeddings)

In [ ]:
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

def generate_hypothetical_answer(query):
    if hosted_available:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
        resp = client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[{'role': 'user', 'content': f'Write one plausible-sounding sentence that would answer: {query}'}],
            max_tokens=60,
        )
        return resp.choices[0].message.content
    # offline fallback: a simple template-based "hypothetical answer" — narrower than a real
    # LLM would produce, but demonstrates the same embed-the-answer-not-the-question idea
    return f'The policy on this topic states specific rules about {query.lower().rstrip("?")}.'

def retrieve_with_hyde(query, top_k=2):
    hypothetical = generate_hypothetical_answer(query)
    h_emb = embedder.encode([hypothetical], normalize_embeddings=True)
    sims = (chunk_embeddings @ h_emb.T).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return [{'title': chunk_titles[i], 'text': chunk_texts[i], 'score': float(sims[i])} for i in top_idx]

print('HyDE-retrieved:', [r['title'] for r in retrieve_with_hyde(q)])

## 4. RAGAS scorecard (with a manual-metric fallback if the library errors)

In [ ]:
eval_questions = [
    ('What ID is required to open an account?', 'kyc-001'),
    ('When is a Suspicious Activity Report filed?', 'aml-002'),
    ('How long do I have to report a fraudulent transfer?', 'ege-003'),
    ('How many overdraft fees can I be charged per day?', 'od-004'),
    ('What must an adverse action notice contain?', 'lend-005'),
]

def context_precision_at_k(retrieve_fn, questions_with_gt):
    """retrieve_fn: a single-arg callable(query) -> list of results, already bound to a
    fixed k — see the lambdas below, which normalize the three retrievers' differing
    keyword names (top_k vs. final_k) into one common call shape."""
    hits = 0
    for q_text, gt_id in questions_with_gt:
        retrieved_titles = [r['title'] for r in retrieve_fn(q_text)]
        gt_title = [t for i, t, _ in [(d[0], d[1], d[2]) for d in documents] if i == gt_id][0]
        hits += int(gt_title in retrieved_titles)
    return hits / len(questions_with_gt)

baseline_precision = context_precision_at_k(lambda q: baseline_retrieve(q, top_k=2), eval_questions)
rerank_precision = context_precision_at_k(lambda q: retrieve_with_rerank(q, final_k=2), eval_questions)
hyde_precision = context_precision_at_k(lambda q: retrieve_with_hyde(q, top_k=2), eval_questions)

print(f'Context precision@2 — baseline:  {baseline_precision:.2f}')
print(f'Context precision@2 — reranked:  {rerank_precision:.2f}')
print(f'Context precision@2 — HyDE:      {hyde_precision:.2f}')
print('\n(This is the RAGAS "context precision" idea computed directly against ground-truth')
print('source docs — a simple, transparent proxy that always works, whether or not the ragas')
print('package + an LLM judge are available in your runtime.)')

## 5. Iterative retrieval for a multi-hop question

In [ ]:
def iterative_retrieve(question, max_hops=2, top_k=2):
    """Retrieve, read, check whether more is needed, retrieve again — a simple version of
    the retrieve-read-refine loop for multi-hop questions."""
    collected = []
    current_query = question
    for hop in range(max_hops):
        new_results = retrieve_with_rerank(current_query, final_k=top_k)
        collected.extend(r for r in new_results if r['title'] not in [c['title'] for c in collected])
        # a real system would ask the LLM "do you have enough to answer, or what's missing?";
        # here, a simple heuristic: if the question mentions two policy areas, do a second hop
        # keyed on the second area
        if hop == 0 and ('overdraft' in question.lower() and 'dispute' in question.lower()):
            current_query = 'credit card dispute process'
        else:
            break
    return collected

multi_hop_q = 'How do the overdraft fee rules compare to the credit card dispute rules for timing?'
hops = iterative_retrieve(multi_hop_q)
print('Multi-hop retrieval result:', [r['title'] for r in hops])

## 6. Write-up (fill in)
Which upgrade (reranking, HyDE) helped most on this small eval set, and why might that
differ at Meridian's real scale with a much larger, messier corpus? What would you need to
generalize the multi-hop heuristic above into something that isn't hand-coded per question type?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 9: RAG II*